# Gold integrated facts

Publish 11 facts at explicit grains after rebuilding and validating the conformed dimensions.

In [ ]:
from pathlib import Path
import importlib
import sys

DATA_PRODUCT_PATH = Path("shared/integrated-test-data/projections/star-schema")
DATA_ROOT_CANDIDATES = [
    Path("/lakehouse/default/Files") / DATA_PRODUCT_PATH,
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / DATA_PRODUCT_PATH,
]

data_root = next(
    (
        candidate
        for candidate in DATA_ROOT_CANDIDATES
        if (candidate / "demo05_support.py").exists()
        and (candidate / "scenario_topology.csv").exists()
    ),
    None,
)
if data_root is None:
    raise FileNotFoundError("Shared star-schema projection was not found.")

if str(data_root) not in sys.path:
    sys.path.insert(0, str(data_root))
import demo05_support as demo

importlib.reload(demo)
sources = demo.load_sources(data_root)
dimensions = demo.build_dimensions(sources)
integrity_results = demo.validate_dimensions(sources, dimensions)
if not integrity_results["passed"].all():
    raise ValueError("Fact publication stopped because dimension integrity checks failed.")

facts = demo.build_facts(sources, dimensions)
observed_counts = {table_name: len(frame) for table_name, frame in facts.items()}
if observed_counts != demo.EXPECTED_FACT_COUNTS:
    raise ValueError(
        f"Fact row counts differ from the governed contract: {observed_counts}"
    )

spark_session = globals().get("spark")
if spark_session is None:
    print("Spark is unavailable; validated Gold facts without publishing Delta tables.")
else:
    for table_name, frame in facts.items():
        (
            spark_session.createDataFrame(demo.spark_compatible_frame(frame))
            .write.mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )

print(observed_counts)